# 04 - TF-IDF Feature Extraction
In this notebook, we convert email text into numerical feature vectors using **TF-IDF** (Term Frequency - Inverse Document Frequency).
- **Part 1**: Understand the mechanics of TF-IDF with a small 3-email example
- **Part 2**: Apply `TfidfVectorizer` to the entire cleaned Enron email dataset
- **Part 3**: Inspect the resulting feature matrix and learned vocabulary

## 1. Understanding TF-IDF with a Small Example

In [ ]:
'''wll learn TF-IDF first with a tiny example, then apply it to your 30,494 emails.'''

In [1]:
''' TfidfVectorizer converts text into numerical TF-IDF features.
 These numbers can later be given to our ML model.'''
from sklearn.feature_extraction.text import TfidfVectorizer

In [13]:
# Three tiny example emails.
# We're using these just to understand how TF-IDF works.

example_emails = [
    "free money offer",
    "free meeting",
    "project meeting"
]

In [14]:
# Create a TF-IDF vectorizer.
# It will look at the words in our emails
# and calculate a TF-IDF value for each word.

vectorizer = TfidfVectorizer()

In [15]:
# Learn the vocabulary from our 3 emails
# and convert the emails into TF-IDF numbers.

tfidf_matrix = vectorizer.fit_transform(example_emails)

In [16]:
# Show the words that TF-IDF learned.

print(vectorizer.get_feature_names_out())

['free' 'meeting' 'money' 'offer' 'project']


In [17]:
# Convert the TF-IDF matrix into a normal-looking array
# so we can see the numbers.

print(tfidf_matrix.toarray())

[[0.4736296  0.         0.62276601 0.62276601 0.        ]
 [0.70710678 0.70710678 0.         0.         0.        ]
 [0.         0.60534851 0.         0.         0.79596054]]


For now, remember:

TF-IDF gives each word a numerical importance based on how often it occurs in an email and how common it is across the collection of emails.

In [ ]:
'''✅ Clean ENRON dataset
✅ Remove duplicate emails
✅ Parse email text
✅ Normalize whitespace
✅ Save cleaned CSV
✅ Text-length EDA
✅ HAM/SPAM distribution
✅ Word tokenization
✅ Word-frequency analysis
✅ HAM vs SPAM frequency comparison
✅ Common-word analysis
✅ SPAM-heavy / HAM-heavy words
✅ Understand why raw frequency isn't enough
✅ Understand TF-IDF concept
✅ Test TF-IDF on 3 tiny example emails'''

## 2. Applying TF-IDF to the Full Dataset

In [2]:
# Note: If running from notebooks directory, path may be '../data/processed/cleaned_emails.csv'
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv("../data/Combined/combined_withOUT_source.csv")

In [3]:
df.head()

,email,label
0,subject riaa site hacked overnight first seen ...,0
1,subject re ilug suse 8 disks thread changed sl...,0
2,subject alliance microcap equity for you alfon...,1
3,subject customizable downld manager unlimited ...,1
4,subject new improved fat burners now with tv f...,1


In [4]:
# Check the shape of the cleaned dataset
print(df.shape)
df.head()

(35806, 2)


,email,label
0,subject riaa site hacked overnight first seen ...,0
1,subject re ilug suse 8 disks thread changed sl...,0
2,subject alliance microcap equity for you alfon...,1
3,subject customizable downld manager unlimited ...,1
4,subject new improved fat burners now with tv f...,1


In [ ]:
"""Give it our actual emails and convert all of it into TF-IDF features
   th fit_transforms does two things:
   1. It learns the vocabulary from our emails.
   2. It converts the emails into TF-IDF numbers based on that vocabulary."""

In [5]:
print(df['label'].value_counts())
print(df['email'].isnull().sum())

label
0    19607
1    16199
Name: count, dtype: int64
0


In [6]:
'''
Lets seperate the input and output targets
'''

X = df['email']
Y = df['label']

In [7]:
"""
Now Train-test split 
split before fitting tf-idf. to prevent data leakage
"""

from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [8]:
print(len(X_test))
print(len(X_train))

7162
28644


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(
    lowercase=True, # convert all text to lowercase
    # stop_words='english', # remove common English words like "the", "and", etc, we will remove it yet 
    max_features=100000, # limit the number of features to 100,000
    ngram_range=(1, 2), # consider both single words (unigrams) and pairs of words (bigrams)
    min_df=5 # ignore words that appear in fewer than 5 emails
)

X_train_tfidf = tfidf.fit_transform(X_train)
# fit_transform learns the vocabulary from the training data and transforms it into TF-IDF features
# we only fit on the training data to avoid data leakage
X_test_tfidf = tfidf.transform(X_test)

In [10]:
#lets check whatr happened
print("Training TF-IDF shape: ", X_train_tfidf.shape) #shows the number of emails and the number of features (words) learned from the training data
print("Test TF-IDF shape: ", X_test_tfidf.shape) # shows the number of emails and the number of features (words) learned from the test data

Training TF-IDF shape:  (28644, 100000)
Test TF-IDF shape:  (7162, 100000)


In [11]:
import joblib

joblib.dump(tfidf, "../models/tfidf_vectorizer.pkl")

['../models/tfidf_vectorizer.pkl']

In [12]:
import os
from scipy.sparse import save_npz

tfidf_output_dir = "../data/processed/tfidf"
os.makedirs(tfidf_output_dir, exist_ok=True)
save_npz(os.path.join(tfidf_output_dir, "X_train_tfidf.npz"), X_train_tfidf)
save_npz(os.path.join(tfidf_output_dir, "X_test_tfidf.npz"), X_test_tfidf)

In [15]:
import pandas as pd

Y_train.to_csv("../data/processed/tfidf/Y_train.csv", index=False)
Y_test.to_csv("../data/processed/tfidf/Y_test.csv", index=False)